In [1]:
pip install mediapipe --upgrade

   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
    --------------------------------------- 0.3/10.6 MB ? eta -:--:--
   -- ------------------------------------- 0.8/10.6 MB 1.5 MB/s eta 0:00:07
   ---- ----------------------------------- 1.3/10.6 MB 1.8 MB/s eta 0:00:06
   ------ --------------------------------- 1.8/10.6 MB 2.0 MB/s eta 0:00:05
   ------- -------------------------------- 2.1/10.6 MB 1.9 MB/s eta 0:00:05
   -------- ------------------------------- 2.4/10.6 MB 1.9 MB/s eta 0:00:05
   ---------- ----------------------------- 2.9/10.6 MB 1.9 MB/s eta 0:00:05
   ------------- -------------------------- 3.7/10.6 MB 2.0 MB/s eta 0:00:04
   --------------- ------------------------ 4.2/10.6 MB 2.2 MB/s eta 0:00:03
   ------------------- -------------------- 5.2/10.6 MB 2.3 MB/s eta 0:00:03
   --------------------- --

In [1]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import time
import numpy as np

# 1. SETUP: Initialize the new FaceLandmarker
base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True, # Essential for 420-grade expressions
    num_faces=1
)
detector = vision.FaceLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    # Convert to MediaPipe Image object
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    
    # 2. RUN DETECTION
    detection_result = detector.detect(mp_image)

    if detection_result.face_landmarks:
        landmarks = detection_result.face_landmarks[0]
        
        # --- Metrics Extraction ---
        # Eye Contact: Using Iris (Indices 468-477 in the task model)
        iris_center = landmarks[468] 
        contact = "Direct" if 0.45 < iris_center.x < 0.55 else "Away"

        # Nodding/Shaking (Simplified Pitch/Yaw from Nose tip index 1)
        nose = landmarks[1]
        nod_status = "Steady"
        if nose.y < 0.4: nod_status = "Nodding Up"
        elif nose.y > 0.6: nod_status = "Nodding Down"

        # Blink Detection (Using Blendshapes for 420-grade accuracy)
        # 9 is 'eyeBlinkLeft', 10 is 'eyeBlinkRight'
        blink_score = detection_result.face_blendshapes[0][9].score
        blink = "Blink" if blink_score > 0.5 else "Open"

        # 3. DISPLAY
        cv2.putText(frame, f"Gaze: {contact}", (20, 50), 1, 2, (0, 255, 0), 2)
        cv2.putText(frame, f"Head: {nod_status}", (20, 100), 1, 2, (255, 0, 0), 2)
        cv2.putText(frame, f"Eyes: {blink}", (20, 150), 1, 2, (0, 0, 255), 2)

    cv2.imshow('Interview Analyzer', frame)
    if cv2.waitKey(5) & 0xFF == 27: break

detector.close()
cap.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 

In [ ]:
print(mp.__version__)

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import numpy as np

# Setup
base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(base_options=base_options, output_face_blendshapes=True, num_faces=1)
detector = vision.FaceLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    if not success: break
    h, w, _ = frame.shape
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    result = detector.detect(mp_image)

    if result.face_landmarks:
        lms = result.face_landmarks[0]
        
        # --- 1. ACCURATE HEAD POSE (Pitch/Yaw) ---
        # Using Normalized distance between Nose (1) and Forehead (10)
        face_top, face_bottom = lms[10], lms[152]
        nose_tip = lms[1]
        
        # Pitch: Vertical position of nose relative to face height
        pitch_ratio = (nose_tip.y - face_top.y) / (face_bottom.y - face_top.y)
        
        head_label = "Steady"
        if pitch_ratio > 0.58: head_label = "Nodding Down" # Calibrate these thresholds
        elif pitch_ratio < 0.42: head_label = "Nodding Up"

        # --- 2. ACCURATE GAZE (Horizontal Iris Ratio) ---
        # Left Eye: Corner(33), Corner(133), Iris Center(468)
        left_iris = lms[468]
        l_corner, r_corner = lms[33], lms[133]
        
        # Calculate ratio: 0.0 (far left) to 1.0 (far right)
        gaze_ratio = (left_iris.x - l_corner.x) / (r_corner.x - l_corner.x)
        
        contact_label = "Away"
        if 0.42 < gaze_ratio < 0.58: contact_label = "EYE CONTACT"

        # --- 3. DISPLAY ---
        color = (0, 255, 0) if contact_label == "EYE CONTACT" else (0, 0, 255)
        cv2.putText(frame, f"Head: {head_label}", (30, 50), 1, 2, (255, 255, 255), 2)
        cv2.putText(frame, f"Gaze: {contact_label}", (30, 100), 1, 2, color, 2)
        cv2.circle(frame, (int(left_iris.x * w), int(left_iris.y * h)), 4, (255, 0, 0), -1)

    cv2.imshow('Pro Interview Analyzer', frame)
    if cv2.waitKey(5) & 0xFF == 27: break

detector.close()
cap.release()


In [3]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2

# 1. SETUP
base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options, 
    output_face_blendshapes=True, # MUST be True for blinks
    num_faces=1
)
detector = vision.FaceLandmarker.create_from_options(options)

cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    if not success: break
    h, w, _ = frame.shape
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    result = detector.detect(mp_image)

    if result.face_landmarks and result.face_blendshapes:
        lms = result.face_landmarks[0]
        blendshapes = result.face_blendshapes[0]
        
        # --- 1. HEAD NODDING (Pitch Ratio) ---
        # 10 is forehead, 152 is chin, 1 is nose tip
        face_top, face_bottom, nose_tip = lms[10], lms[152], lms[1]
        pitch_ratio = (nose_tip.y - face_top.y) / (face_bottom.y - face_top.y)
        
        head_label = "Steady"
        if pitch_ratio > 0.62: head_label = "Nodding Down" # Increased threshold to fix "Always Down"
        elif pitch_ratio < 0.38: head_label = "Nodding Up"

        # --- 2. EYE CONTACT (Gaze Ratio) ---
        # Left Eye: Corner(33), Corner(133), Iris Center(468)
        left_iris = lms[468]
        l_corner, r_corner = lms[33], lms[133]
        gaze_ratio = (left_iris.x - l_corner.x) / (r_corner.x - l_corner.x)
        
        contact_label = "Looking Away"
        if 0.40 < gaze_ratio < 0.60: contact_label = "EYE CONTACT"

        # --- 3. EYE BLINK (Blendshapes) ---
        # eyeBlinkLeft is index 9, eyeBlinkRight is index 10 in MediaPipe Blendshapes
        left_blink = blendshapes[9].score
        right_blink = blendshapes[10].score
        blink_label = "Blink!" if (left_blink > 0.5 or right_blink > 0.5) else "Open"

        # --- 4. DISPLAY ---
        cv2.putText(frame, f"Gaze: {contact_label}", (30, 50), 1, 2, (0, 255, 0) if contact_label == "EYE CONTACT" else (0, 0, 255), 2)
        cv2.putText(frame, f"Head: {head_label}", (30, 100), 1, 2, (255, 255, 255), 2)
        cv2.putText(frame, f"Eyes: {blink_label}", (30, 150), 1, 2, (255, 255, 0), 2)

    cv2.imshow('Final Interview Analyzer', frame)
    if cv2.waitKey(5) & 0xFF == 27: break

detector.close()
cap.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 

In [ ]:
nltk.download('punkt')
